In [ ]:
!pwd

In [ ]:
root_folber = '/home/govlept1004/jupyter_home/tank_project'
!mkdir $root_folber/detection

In [ ]:
!mkdir -p detection/tank_dataset

!mkdir -p detection/tank_dataset/images
!mkdir -p detection/tank_dataset/labels

!mkdir -p detection/tank_dataset/images/train
!mkdir -p detection/tank_dataset/images/val

!mkdir -p detection/tank_dataset/labels/train
!mkdir -p detection/tank_dataset/labels/val

In [ ]:
import zipfile
import os
import random
from pathlib import Path
import shutil

# zip 파일 경로
image_zip_path = '/home/govlept1004/jupyter_home/tank_project/detection/image.zip'
label_zip_path = '/home/govlept1004/jupyter_home/tank_project/detection/yolo.txt.zip'

# 임시 압축 해제 디렉토리
temp_image_dir = '/home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/temp_images'
temp_label_dir = '/home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/temp_labels'

# 최종 목적지
train_img_dir = '/home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/images/train'
train_lbl_dir = '/home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/labels/train'
val_img_dir = '/home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/images/val'
val_lbl_dir = '/home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/labels/val'

# 폴더 생성
for folder in [temp_image_dir, temp_label_dir, train_img_dir, train_lbl_dir, val_img_dir, val_lbl_dir]:
    os.makedirs(folder, exist_ok=True)

# 이미지 압축 해제
with zipfile.ZipFile(image_zip_path, 'r') as zip_ref:
    zip_ref.extractall(temp_image_dir)

# 라벨 압축 해제
with zipfile.ZipFile(label_zip_path, 'r') as zip_ref:
    zip_ref.extractall(temp_label_dir)

# 이미지 목록 정렬
image_files = sorted([f for f in os.listdir(temp_image_dir) if f.endswith(('.jpg', '.png'))])
total = len(image_files)
val_count = int(total * 0.1)

# 무작위 섞기
random.shuffle(image_files)
val_images = image_files[:val_count]
train_images = image_files[val_count:]

# 이동 함수
def move_files(image_list, img_src, lbl_src, img_dst, lbl_dst):
    for img_file in image_list:
        base = Path(img_file).stem
        label_file = base + '.txt'

        # 이미지 이동
        shutil.move(os.path.join(img_src, img_file), os.path.join(img_dst, img_file))

        # 라벨이 있다면 이동
        label_path = os.path.join(lbl_src, label_file)
        if os.path.exists(label_path):
            shutil.move(label_path, os.path.join(lbl_dst, label_file))

# 실제 이동
move_files(train_images, temp_image_dir, temp_label_dir, train_img_dir, train_lbl_dir)
move_files(val_images, temp_image_dir, temp_label_dir, val_img_dir, val_lbl_dir)

print(f"✅ 완료: {len(train_images)}개는 train으로, {len(val_images)}개는 val로 이동했습니다.")


In [ ]:
import os
import shutil

# 최상위 폴더 경로 (압축 풀린 경로)
extracted_label_dir = '/home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/temp_labels/yolo.txt'

# 실제 사용할 최상위 폴더
target_label_dir = '/home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/temp_labels'

# 폴더 안에 파일들만 한 단계 위로 이동
for fname in os.listdir(extracted_label_dir):
    src = os.path.join(extracted_label_dir, fname)
    dst = os.path.join(target_label_dir, fname)
    shutil.move(src, dst)

# 빈 폴더 삭제
os.rmdir(extracted_label_dir)


In [ ]:
import os
import shutil

# 최상위 폴더 경로 (압축 풀린 경로)
extracted_label_dir = '/home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/temp_images/image'

# 실제 사용할 최상위 폴더
target_label_dir = '/home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/temp_images'

# 폴더 안에 파일들만 한 단계 위로 이동
for fname in os.listdir(extracted_label_dir):
    src = os.path.join(extracted_label_dir, fname)
    dst = os.path.join(target_label_dir, fname)
    shutil.move(src, dst)

# 빈 폴더 삭제
os.rmdir(extracted_label_dir)

In [ ]:
# 이미지 목록 정렬
image_files = sorted([f for f in os.listdir(temp_image_dir) if f.endswith(('.jpg', '.png'))])
total = len(image_files)
val_count = int(total * 0.2)

# 무작위 섞기
random.shuffle(image_files)
val_images = image_files[:val_count]
train_images = image_files[val_count:]

# 이동 함수
def move_files(image_list, img_src, lbl_src, img_dst, lbl_dst):
    for img_file in image_list:
        base = Path(img_file).stem
        label_file = base + '.txt'

        # 이미지 이동
        shutil.move(os.path.join(img_src, img_file), os.path.join(img_dst, img_file))

        # 라벨이 있다면 이동
        label_path = os.path.join(lbl_src, label_file)
        if os.path.exists(label_path):
            shutil.move(label_path, os.path.join(lbl_dst, label_file))

# 실제 이동
move_files(train_images, temp_image_dir, temp_label_dir, train_img_dir, train_lbl_dir)
move_files(val_images, temp_image_dir, temp_label_dir, val_img_dir, val_lbl_dir)

print(f"✅ 완료: {len(train_images)}개는 train으로, {len(val_images)}개는 val로 이동했습니다.")


In [ ]:
from ultralytics import YOLO

# 재학습
model = YOLO('yolov8m.pt')

result = model.train(
    data='/home/govlept1004/jupyter_home/tank_project/detection/data.yaml',
    epochs=300,
    imgsz=640,
    batch=16,             # 모델이 더 크므로 줄이는 것이 안전
    device=1,
    workers=4,
    name='yolov8m_tank' 
)

In [ ]:
from ultralytics import YOLO

# 2차 학습 (best.pt 불러와서 파인튜닝)
model2 = YOLO("/home/govlept1004/jupyter_home/tank_project/runs/detect/yolov8m_tank4/weights/best.pt")
result2 = model2.train(
    data="/home/govlept1004/jupyter_home/tank_project/detection/data.yaml",
    epochs=200,
    imgsz=640,
    batch=16,        
    device=1,
    workers=4,
    name='yolov8m_tank_refined'  
)

In [ ]:
!find . -name "*.zip"

In [ ]:
!unzip detection/test_image.zip -d detection/test_image

In [ ]:
# 재학습된 모델 불러오기
model = YOLO('/home/govlept1004/jupyter_home/tank_project/runs/detect/yolov8s_tank/weights/best.pt')

results = model('/home/govlept1004/jupyter_home/tank_project/detection/test_image', save=True)

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

%matplotlib inline

# 예측 이미지가 저장된 폴더
img_dir = '/home/govlept1004/jupyter_home/tank_project/runs/detect/predict'  # 예: runs/detect/predict2

# 이미지 파일 리스트
img_files = [f for f in os.listdir(img_dir) if f.endswith(('.jpg'))]

# 여러 이미지 출력
plt.figure(figsize=(15, 10))

for i, file in enumerate(img_files[:]):
    img = cv2.imread(os.path.join(img_dir, file))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.title(file)
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
!pwd

In [ ]:
!unzip detection/Image-20250602T082130Z-1-001.zip -d detection/tank_dataset/images/train

In [ ]:
!unzip detection/LabelText-20250602T082127Z-1-001.zip -d detection/tank_dataset/labels/train

In [ ]:
!unzip detection/od_img_4481.zip -d detection/tank_dataset/images/train

In [ ]:
!unzip detection/od_img_4500.zip -d detection/tank_dataset/labels/train

In [ ]:
mv detection/tank_dataset/images/train/Image/* /home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/images/train/

In [ ]:
mv detection/tank_dataset/labels/train/LabelText/* /home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/labels/train/

In [18]:
!unzip detection/Image_5000_no_3501~4000.zip -d detection/

Archive:  detection/Image_5000_no_3501~4000.zip
   creating: detection/Image/
   creating: detection/Image/images/
  inflating: detection/Image/images/od_img_1.png  
  inflating: detection/Image/images/od_img_10.png  
  inflating: detection/Image/images/od_img_100.png  
  inflating: detection/Image/images/od_img_1000.png  
  inflating: detection/Image/images/od_img_1001.jpg  
  inflating: detection/Image/images/od_img_1002.jpg  
  inflating: detection/Image/images/od_img_1003.jpg  
  inflating: detection/Image/images/od_img_1004.jpg  
  inflating: detection/Image/images/od_img_1005.jpg  
  inflating: detection/Image/images/od_img_1006.jpg  
  inflating: detection/Image/images/od_img_1007.jpg  
  inflating: detection/Image/images/od_img_1008.jpg  
  inflating: detection/Image/images/od_img_1009.jpg  
  inflating: detection/Image/images/od_img_101.png  
  inflating: detection/Image/images/od_img_1010.jpg  
  inflating: detection/Image/images/od_img_1011.jpg  
  inflating: detection/Image

In [ ]:
import os
import shutil

# 경로 설정
src_dir = "/home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/image"
img_dst = "/home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/images/train"
txt_dst = "/home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/labels/train"

# 이동 함수 정의
def move_files(src_dir, dst_dir, ext):
    moved = 0
    for filename in os.listdir(src_dir):
        if filename.endswith(ext):
            src_file = os.path.join(src_dir, filename)
            dst_file = os.path.join(dst_dir, filename)
            if not os.path.exists(dst_file):
                shutil.move(src_file, dst_file)
                print(f"Moved {ext.upper()} file: {filename}")
                moved += 1
    return moved

# PNG 파일 이동
png_moved = move_files(src_dir, img_dst, ".jpg")

# TXT 파일 이동
txt_moved = move_files(src_dir, txt_dst, ".txt")

print(f"\n총 이동된 파일 수:")
print(f"PNG: {png_moved}개")
print(f"TXT: {txt_moved}개")


In [ ]:
import os
import shutil
import random

# 디렉토리 설정
image_train_dir = '/home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/images/train'
image_val_dir = '/home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/images/val'
label_train_dir = '/home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/labels/train'
label_val_dir = '/home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/labels/val'

# 이미지 확장자 목록 (필요 시 추가 가능)
image_exts = ['.png','.jpg']

# 이미지 파일 목록만 가져오기
train_images = [f for f in os.listdir(image_train_dir)
                if os.path.splitext(f)[1].lower() in image_exts]

val_images = [f for f in os.listdir(image_val_dir)
              if os.path.splitext(f)[1].lower() in image_exts]

# 전체 수 계산 및 8:2 목표 설정
total_images = len(train_images) + len(val_images)
target_val_count = int(total_images * 0.2)
need_to_move = target_val_count - len(val_images)

# 이동할 이미지 파일 무작위 선택
files_to_move = random.sample(train_images, need_to_move)

# 파일 이동
for img_file in files_to_move:
    base_name, ext = os.path.splitext(img_file)
    label_file = base_name + '.txt'

    # 이미지 경로
    img_src = os.path.join(image_train_dir, img_file)
    img_dst = os.path.join(image_val_dir, img_file)

    # 라벨 경로
    label_src = os.path.join(label_train_dir, label_file)
    label_dst = os.path.join(label_val_dir, label_file)

    # 이동
    shutil.move(img_src, img_dst)
    if os.path.exists(label_src):
        shutil.move(label_src, label_dst)

print(f"{need_to_move}개의 이미지와 라벨 파일을 train → val로 이동 완료.")


In [19]:
import os
import random
import shutil
from pathlib import Path

# 원본 이미지 및 라벨 경로
image_dir = "/home/govlept1004/jupyter_home/tank_project/detection/Image/images"  # 원본 이미지 폴더
label_dir = "/home/govlept1004/jupyter_home/tank_project/detection/Image/labels"  # 원본 라벨 폴더

# 분할 대상 폴더 생성
for split in ['train', 'val']:
    os.makedirs(os.path.join(image_dir, split), exist_ok=True)
    os.makedirs(os.path.join(label_dir, split), exist_ok=True)

# 이미지 파일 목록 수집 (확장자 자동 인식)
image_exts = ['.jpg', '.png']
image_files = [f for f in os.listdir(image_dir) if Path(f).suffix in image_exts]

# 셔플 후 8:2 분할
random.seed(42)
random.shuffle(image_files)
split_idx = int(0.8 * len(image_files))
train_files = image_files[:split_idx]
val_files = image_files[split_idx:]

def move_data(file_list, split):
    for img_file in file_list:
        img_path = os.path.join(image_dir, img_file)
        label_file = Path(img_file).with_suffix(".txt")
        label_path = os.path.join(label_dir, label_file)

        # 대상 경로
        img_dst = os.path.join(image_dir, split, img_file)
        label_dst = os.path.join(label_dir, split, label_file)

        # 이동
        shutil.move(img_path, img_dst)
        if os.path.exists(label_path):
            shutil.move(label_path, label_dst)
        else:
            print(f"⚠️ 라벨 없음: {label_path}")

# 실제 이동
move_data(train_files, "train")
move_data(val_files, "val")

print(f"✅ 완료: {len(train_files)}개 train, {len(val_files)}개 val로 분할 완료")


✅ 완료: 3600개 train, 900개 val로 분할 완료


In [ ]:
import os
from collections import Counter

# 라벨 폴더 경로 설정
label_dir = '/home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/labels/train'  # train 또는 전체 경로

# 모든 .txt 파일에서 클래스 번호만 추출
class_counts = Counter()

for file in os.listdir(label_dir):
    if file.endswith('.txt'):
        with open(os.path.join(label_dir, file), 'r') as f:
            for line in f:
                class_id = int(line.split()[0])
                class_counts[class_id] += 1

# 결과 출력
total = sum(class_counts.values())
for class_id, count in sorted(class_counts.items()):
    ratio = count / total * 100
    print(f'Class {class_id}: {count}개 ({ratio:.2f}%)')

print(f'\n총 객체 수: {total}')


In [20]:
from ultralytics import YOLO

# 재학습
model = YOLO('yolov8n.pt')

# 재학습
result = model.train(
    data='/home/govlept1004/jupyter_home/tank_project/detection/data.yaml',
    epochs=500,
    imgsz=640,
    batch=32,
    device=0,
    workers=4,
    name='yolov8_tank'
)

New https://pypi.org/project/ultralytics/8.3.149 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.146 🚀 Python-3.10.16 torch-2.2.0 CUDA:0 (NVIDIA GeForce RTX 3070 Ti Laptop GPU, 8192MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/govlept1004/jupyter_home/tank_project/detection/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, n

train: Scanning /home/govlept1004/jupyter_home/tank_project/detection/Image/labels/train... 3600 images, 528 backgrounds, 0 corrupt: 100%|██████████| 3600/3600 [00:13<00:00, 257.39it/s]

train: /home/govlept1004/jupyter_home/tank_project/detection/Image/images/train/od_img_1044.jpg: 1 duplicate labels removed
train: /home/govlept1004/jupyter_home/tank_project/detection/Image/images/train/od_img_1062.jpg: 1 duplicate labels removed
train: /home/govlept1004/jupyter_home/tank_project/detection/Image/images/train/od_img_1099.jpg: 1 duplicate labels removed
train: /home/govlept1004/jupyter_home/tank_project/detection/Image/images/train/od_img_1150.jpg: 1 duplicate labels removed
train: /home/govlept1004/jupyter_home/tank_project/detection/Image/images/train/od_img_1222.jpg: 1 duplicate labels removed
train: /home/govlept1004/jupyter_home/tank_project/detection/Image/images/train/od_img_1240.jpg: 1 duplicate labels removed
train: /home/govlept1004/jupyter_home/tank_project/detection/Image/images/train/od_img_1253.jpg: 1 duplicate labels removed
train: /home/govlept1004/jupyter_home/tank_project/detection/Image/images/train/od_img_1394.jpg: 1 duplicate labels removed
train: /

train: New cache created: /home/govlept1004/jupyter_home/tank_project/detection/Image/labels/train.cache
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 221.0±89.1 MB/s, size: 397.0 KB)


val: Scanning /home/govlept1004/jupyter_home/tank_project/detection/Image/labels/val... 900 images, 137 backgrounds, 0 corrupt: 100%|██████████| 900/900 [00:03<00:00, 241.27it/s]

...

                   all        900       2476      0.768      0.607      0.681      0.393



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      5/500      4.04G      1.418      1.141      1.139        159        640:  63%|██████▎   | 71/113 [00:16<00:09,  4.28it/s]


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/govlept1004/anaconda3/envs/keras_env/lib/python3.10/site-packages/IPython/core/interactiveshell.py", line 3508, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_10690/3654141305.py", line 7, in <module>
    result = model.train(
  File "/home/govlept1004/anaconda3/envs/keras_env/lib/python3.10/site-packages/ultralytics/engine/model.py", line 797, in train
    self.trainer.train()
  File "/home/govlept1004/anaconda3/envs/keras_env/lib/python3.10/site-packages/ultralytics/engine/trainer.py", line 227, in train
    self._do_train(world_size)
  File "/home/govlept1004/anaconda3/envs/keras_env/lib/python3.10/site-packages/ultralytics/engine/trainer.py", line 406, in _do_train
    loss, self.loss_items = self.model(batch)
  File "/home/govlept1004/anaconda3/envs/keras_env/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1511, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)

In [ ]:
import torch
print(torch.__version__)
import tensorflow as tf
print(tf.__version__)


In [ ]:
result = model.train(
    data='/home/govlept1004/jupyter_home/tank_project/detection/data.yaml', epochs=200, imgsz=640, batch=16, optimizer='SGD',     
    lr0=0.01,            
    weight_decay=0.0005, 
    augment=True,        
    device=1,            
    workers=4,           
    name='yolov8n_tank'  
)



yolo detect train model=yolov8n.pt data=/home/govlept1004/jupyter_home/tank_project/detection/data.yaml epochs=500 imgsz=640 batch=16 device=0 workers=4 optimizer='SGD' name=yolov8n_tank

In [ ]:
yolo model.train(data='/home/govlept1004/jupyter_home/tank_project/detection/data.yaml', epochs=200, imgsz=640, batch=8, optimizer='SGD', lr0=0.01, weight_decay=0.0005, augment=True, device='cuda:0', workers=2, name='yolov8n_tank')


In [17]:
from ultralytics import YOLO

# 재학습
model = YOLO('/home/govlept1004/jupyter_home/tank_project/runs/detect/yolov8n_tank5/weights/last.pt')

# 재학습
result = model.train(
    data='/home/govlept1004/jupyter_home/tank_project/detection/data.yaml',
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,
    workers=4,
    patience=500,
    name='yolov8_tank'
)

New https://pypi.org/project/ultralytics/8.3.149 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.146 🚀 Python-3.10.16 torch-2.2.0 CUDA:0 (NVIDIA GeForce RTX 3070 Ti Laptop GPU, 8192MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/govlept1004/jupyter_home/tank_project/detection/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/govlept1004/jupyter_home/tank_project/runs/detect/yolo

train: Scanning /home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/labels/train.cache... 3440 images, 249 backgrounds, 457 corrupt: 100%|██████████| 3440/3440 [00:00<?, ?it/s]

train: /home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/images/train/od_img_10.png: ignoring corrupt image/label: Label class 5 exceeds dataset class count 3. Possible class labels are 0-2
train: /home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/images/train/od_img_100.png: ignoring corrupt image/label: Label class 5 exceeds dataset class count 3. Possible class labels are 0-2
train: /home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/images/train/od_img_101.png: ignoring corrupt image/label: Label class 4 exceeds dataset class count 3. Possible class labels are 0-2
train: /home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/images/train/od_img_103.png: ignoring corrupt image/label: Label class 4 exceeds dataset class count 3. Possible class labels are 0-2
train: /home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/images/train/od_img_104.png: ignoring corrupt image/label: Label class 5 exceeds dataset class 

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 288.1±66.0 MB/s, size: 485.4 KB)


val: Scanning /home/govlept1004/jupyter_home/tank_project/detection/tank_dataset/labels/val.cache... 822 images, 35 backgrounds, 275 corrupt: 100%|██████████| 822/822 [00:00<?, ?it/s]

...

    100/100      4.03G     0.7245     0.3918     0.8774         11        640: 100%|██████████| 94/94 [00:19<00:00,  4.76it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 9/9 [00:02<00:00,  3.47it/s]

                   all        547       1541      0.828      0.692      0.723      0.511



100 epochs completed in 0.644 hours.
Optimizer stripped from runs/detect/yolov8_tank3/weights/last.pt, 6.3MB
Optimizer stripped from runs/detect/yolov8_tank3/weights/best.pt, 6.3MB

Validating runs/detect/yolov8_tank3/weights/best.pt...
Ultralytics 8.3.146 🚀 Python-3.10.16 torch-2.2.0 CUDA:0 (NVIDIA GeForce RTX 3070 Ti Laptop GPU, 8192MiB)
Model summary (fused): 72 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 9/9 [00:03<00:00,  2.39it/s]


                   all        547       1541      0.806      0.757      0.828      0.557
                E_Tank        279        504      0.863      0.776      0.888      0.649
                   Car        266        702      0.718      0.811      0.803      0.511
                 Human        216        335      0.836      0.684      0.792      0.511
Speed: 0.2ms preprocess, 1.2ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to runs/detect/yolov8_tank3


In [1]:
from ultralytics import YOLO

# 모델 불러오기 (사전 학습된 yolov8n)
model = YOLO('yolov8n.pt')

# 학습 실행
model.train(
    data='/home/govlept1004/jupyter_home/tank_project/detection/data.yaml',
    epochs=500,
    imgsz=640,
    batch=32,
    device=0,        # CUDA:0 사용 (GPU)
    workers=4,       # DataLoader 병렬 처리 수
    name='yolov8_tank'
)

New https://pypi.org/project/ultralytics/8.3.149 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.146 🚀 Python-3.10.16 torch-2.2.0 CUDA:0 (NVIDIA GeForce RTX 3070 Ti Laptop GPU, 8192MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/govlept1004/jupyter_home/tank_project/detection/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, n

train: Scanning /home/govlept1004/jupyter_home/tank_project/detection/Image/labels/train.cache... 3600 images, 5

train: /home/govlept1004/jupyter_home/tank_project/detection/Image/images/train/od_img_1044.jpg: 1 duplicate labels removed
train: /home/govlept1004/jupyter_home/tank_project/detection/Image/images/train/od_img_1062.jpg: 1 duplicate labels removed
train: /home/govlept1004/jupyter_home/tank_project/detection/Image/images/train/od_img_1099.jpg: 1 duplicate labels removed
train: /home/govlept1004/jupyter_home/tank_project/detection/Image/images/train/od_img_1150.jpg: 1 duplicate labels removed
train: /home/govlept1004/jupyter_home/tank_project/detection/Image/images/train/od_img_1222.jpg: 1 duplicate labels removed
train: /home/govlept1004/jupyter_home/tank_project/detection/Image/images/train/od_img_1240.jpg: 1 duplicate labels removed
train: /home/govlept1004/jupyter_home/tank_project/detection/Image/images/train/od_img_1253.jpg: 1 duplicate labels removed
train: /home/govlept1004/jupyter_home/tank_project/detection/Image/images/train/od_img_1394.jpg: 1 duplicate labels removed
train: /

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 365.8±190.5 MB/s, size: 397.0 KB)


val: Scanning /home/govlept1004/jupyter_home/tank_project/detection/Image/labels/val.cache... 900 images, 137 ba

...

                   all        900       2476      0.912      0.873      0.907      0.663
EarlyStopping: Training stopped early as no improvement observed in last 100 epochs. Best results observed at epoch 327, best model saved as best.pt.
To update EarlyStopping(patience=100) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



427 epochs completed in 11.085 hours.
Optimizer stripped from runs/detect/yolov8_tank7/weights/last.pt, 6.3MB
Optimizer stripped from runs/detect/yolov8_tank7/weights/best.pt, 6.3MB

Validating runs/detect/yolov8_tank7/weights/best.pt...
Ultralytics 8.3.146 🚀 Python-3.10.16 torch-2.2.0 CUDA:0 (NVIDIA GeForce RTX 3070 Ti Laptop GPU, 8192MiB)
Model summary (fused): 72 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 15/15


                   all        900       2476      0.916      0.865      0.908      0.668
                E_Tank        413        564      0.899        0.9      0.924      0.697
                   Car        426       1358      0.922      0.902      0.936      0.687
                 Human        319        554      0.928      0.792      0.863      0.621
Speed: 0.3ms preprocess, 3.9ms inference, 0.0ms loss, 4.5ms postprocess per image
Results saved to runs/detect/yolov8_tank7


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f3d257253c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04